# Terafab Decision Twin — Comparative 2026–2030 Scenarios

This Google Colab notebook installs the public `terafab-decision-twin` repository, validates three 2026–2030 scenarios, runs the simulations, and generates a single-page comparative infographic for policymakers and investors.

Scenarios used:
- `multi_year_2026_2030.json` — reference/base case already in the repo
- `worst_case_2026_2030.json` — near-failure stress case
- `best_case_2026_2030.json` — milestone-support case

**Repository:** https://github.com/KNOWDYN/terafab-decision-twin


In [ ]:
# Clone the public repo and install the package.
!git clone https://github.com/KNOWDYN/terafab-decision-twin.git
%cd /content/terafab-decision-twin
!pip -q install .


## Upload reminder

Before running the scenario cells below, make sure `worst_case_2026_2030.json` and `best_case_2026_2030.json` have been uploaded into the repository folder:

```text
/content/terafab-decision-twin/scenarios/
```

If you already committed them to GitHub before opening this notebook, they will already be present after the clone.


In [ ]:
from pathlib import Path
import json
import math
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from matplotlib.ticker import FuncFormatter

from terafab_decision_twin.schema import load_scenario, validate_scenario
from terafab_decision_twin.engine import run_scenario

repo_dir = Path('/content/terafab-decision-twin')
scenario_dir = repo_dir / 'scenarios'
scenario_map = {
    'Reference': scenario_dir / 'multi_year_2026_2030.json',
    'Worst case': scenario_dir / 'worst_case_2026_2030.json',
    'Best case': scenario_dir / 'best_case_2026_2030.json',
}

for label, path in scenario_map.items():
    print(label, '->', path, 'exists:', path.exists())


### Optional: upload the two new scenario files if they are not yet in the cloned repo

If the `exists:` check above shows `False` for either new file, run the cell below and upload:
- `worst_case_2026_2030.json`
- `best_case_2026_2030.json`


In [ ]:
# Optional upload helper for Colab.
from google.colab import files
uploaded = files.upload()
for name, content in uploaded.items():
    target = scenario_dir / name
    target.write_bytes(content)
    print('Saved', target)


In [ ]:
# Validate scenarios before simulation.
validation_rows = []
for label, path in scenario_map.items():
    scenario = load_scenario(path)
    errors = validate_scenario(scenario)
    validation_rows.append({'scenario': label, 'valid': len(errors) == 0, 'errors': '; '.join(errors) if errors else ''})

validation_df = pd.DataFrame(validation_rows)
validation_df


In [ ]:
# Run all three scenarios.
results = {}
for label, path in scenario_map.items():
    scenario = load_scenario(path)
    results[label] = run_scenario(scenario)

summary_rows = []
gate_rows = []
for label, result in results.items():
    s = result['summary']
    summary_rows.append({
        'scenario': label,
        'average_site_load_MW': s['average_site_load_MW'],
        'firm_capacity_margin_MW': s['minimum_firm_capacity_margin_MW'],
        'heat_rejection_margin_MW': s['minimum_heat_rejection_margin_MW'],
        'water_withdrawal_margin_m3_per_day': s['minimum_water_withdrawal_margin_m3_per_day'],
        'effective_yield': s['average_effective_yield'],
        'total_cost_USD_bn': s['total_cost_USD'] / 1e9,
        'cost_per_good_die_USD': s['cost_per_good_die_USD'],
        'emissions_MtCO2': s['emissions_tCO2'] / 1e6,
        'legitimacy_margin': s['legitimacy_margin'],
        'governance_risk_index': s['maximum_governance_risk_index'],
        'recommended_action': s['recommended_phase_action'],
        'passed': result['passed'],
        'failed_gate_count': sum(1 for g in result['gates'] if not g['passed'])
    })
    for gate in result['gates']:
        gate_rows.append({
            'scenario': label,
            'gate': gate['name'],
            'passed': gate['passed'],
            'severity': gate['severity'],
            'margin': gate['margin']
        })

summary_df = pd.DataFrame(summary_rows).set_index('scenario')
gates_df = pd.DataFrame(gate_rows)
summary_df


In [ ]:
# Generate a one-page infographic for policymakers and investors.

def money_fmt(x, pos):
    return f'${x:.0f}B'

def num_fmt(x, pos):
    return f'{x:.0f}'

def pct_fmt(x, pos):
    return f'{100*x:.0f}%'

order = ['Worst case', 'Reference', 'Best case']
plot_df = summary_df.loc[order]

gate_pivot = gates_df.assign(value=gates_df['passed'].astype(int)).pivot(index='gate', columns='scenario', values='value')[order]

fig = plt.figure(figsize=(14, 18), constrained_layout=True)
gs = GridSpec(5, 2, figure=fig, height_ratios=[0.8, 1.1, 1.1, 1.1, 1.25])

ax_title = fig.add_subplot(gs[0, :])
ax_title.axis('off')
title = 'Terafab Decision Twin | Comparative 2026–2030 Scenario Outlook'
subtitle = ('Reference vs. worst-case near-failure stress vs. best-case milestone-support case. '
            'All values shown are model outputs from evidence-gated scenario assumptions, not verified Terafab operating facts.')
ax_title.text(0.0, 0.85, title, fontsize=20, fontweight='bold', ha='left', va='top')
ax_title.text(0.0, 0.48, subtitle, fontsize=11, ha='left', va='top', wrap=True)
status_lines = []
for label in order:
    row = plot_df.loc[label]
    status_lines.append(f"{label}: action = {row['recommended_action']}, failed gates = {int(row['failed_gate_count'])}")
ax_title.text(0.0, 0.10, '\n'.join(status_lines), fontsize=11, ha='left', va='bottom')

ax1 = fig.add_subplot(gs[1, 0])
plot_df[['average_site_load_MW', 'firm_capacity_margin_MW']].plot(kind='bar', ax=ax1)
ax1.set_title('Power demand and firm-capacity margin')
ax1.set_ylabel('MW')
ax1.tick_params(axis='x', rotation=0)
ax1.axhline(0, linewidth=1)

ax2 = fig.add_subplot(gs[1, 1])
plot_df[['heat_rejection_margin_MW', 'water_withdrawal_margin_m3_per_day']].rename(columns={'heat_rejection_margin_MW':'Cooling margin (MW)','water_withdrawal_margin_m3_per_day':'Water permit margin (m3/day)'}).plot(kind='bar', ax=ax2)
ax2.set_title('Thermodynamic and water headroom')
ax2.tick_params(axis='x', rotation=0)
ax2.axhline(0, linewidth=1)

ax3 = fig.add_subplot(gs[2, 0])
plot_df[['effective_yield', 'legitimacy_margin']].plot(kind='bar', ax=ax3)
ax3.set_title('Yield and public legitimacy')
ax3.yaxis.set_major_formatter(FuncFormatter(pct_fmt))
ax3.tick_params(axis='x', rotation=0)
ax3.axhline(0, linewidth=1)

ax4 = fig.add_subplot(gs[2, 1])
plot_df[['governance_risk_index']].plot(kind='bar', ax=ax4)
ax4.set_title('Governance risk index (lower is better)')
ax4.set_ylabel('Index')
ax4.tick_params(axis='x', rotation=0)
ax4.axhline(0, linewidth=1)

ax5 = fig.add_subplot(gs[3, 0])
plot_df[['total_cost_USD_bn', 'cost_per_good_die_USD']].rename(columns={'total_cost_USD_bn':'Total cost (USD bn)','cost_per_good_die_USD':'Cost per good die (USD)'}).plot(kind='bar', ax=ax5)
ax5.set_title('Economic exposure')
ax5.tick_params(axis='x', rotation=0)
ax5.axhline(0, linewidth=1)

ax6 = fig.add_subplot(gs[3, 1])
plot_df[['emissions_MtCO2']].rename(columns={'emissions_MtCO2':'Emissions (MtCO2)'}).plot(kind='bar', ax=ax6)
ax6.set_title('Emissions burden')
ax6.tick_params(axis='x', rotation=0)
ax6.axhline(0, linewidth=1)

ax7 = fig.add_subplot(gs[4, 0])
ax7.set_title('Gate matrix (1 = pass, 0 = fail)')
im = ax7.imshow(gate_pivot.values, aspect='auto', interpolation='nearest')
ax7.set_xticks(range(len(order)))
ax7.set_xticklabels(order, rotation=0)
ax7.set_yticks(range(len(gate_pivot.index)))
ax7.set_yticklabels(gate_pivot.index)
for i in range(gate_pivot.shape[0]):
    for j in range(gate_pivot.shape[1]):
        ax7.text(j, i, int(gate_pivot.iloc[i, j]), ha='center', va='center', fontsize=9)

ax8 = fig.add_subplot(gs[4, 1])
ax8.axis('off')

def takeaway(label):
    row = plot_df.loc[label]
    return (
        f"{label}\n"
        f"• Action: {row['recommended_action']}\n"
        f"• Failed gates: {int(row['failed_gate_count'])}\n"
        f"• Yield: {row['effective_yield']:.1%}\n"
        f"• Total cost: ${row['total_cost_USD_bn']:.2f}B\n"
        f"• Emissions: {row['emissions_MtCO2']:.2f} MtCO2\n"
        f"• Legitimacy margin: {row['legitimacy_margin']:.2f}\n"
    )

text_block = (
    'Decision takeaways\n\n' +
    takeaway('Worst case') + '\n' +
    takeaway('Reference') + '\n' +
    takeaway('Best case') +
    '\nInterpretation:\n'
    '• Worst case highlights which constraints become binding first.\n'
    '• Reference case shows a monitored middle path.\n'
    '• Best case indicates the performance envelope if infrastructure, governance, and regulation remain supportive.'
)
ax8.text(0, 1, text_block, ha='left', va='top', fontsize=10)

plt.suptitle('Policy and investor infographic', y=1.01, fontsize=12)
plt.show()


In [ ]:
# Save key outputs if needed.
summary_df.to_csv('comparative_summary_2026_2030.csv')
gates_df.to_csv('comparative_gates_2026_2030.csv', index=False)
fig.savefig('comparative_infographic_2026_2030.png', dpi=200, bbox_inches='tight')
print('Saved comparative_summary_2026_2030.csv, comparative_gates_2026_2030.csv, comparative_infographic_2026_2030.png')
